# 003. Extract dark-matter candidates

Canonical candidate-extraction stage for the paper pipeline.


## Purpose, inputs, and outputs

**Purpose**
- Clean the canonical paper abstracts for model extraction.
- Identify dark-matter candidate mentions.
- Build the lean paper-level and candidate-level outputs used by the manuscript figures.

**Reads**
- `data/papers.parquet`
- `data/paper_arxiv_classes.parquet`

**Writes**
- `data/papers_with_dm_models.parquet`
- `data/dm_model_candidates_long.parquet`
- `code/stage-outputs/003-extract-dm-candidates/dm_model_mentions.parquet`
- `code/stage-outputs/003-extract-dm-candidates/dm_model_counts_by_year.parquet`


In [1]:
from pathlib import Path

import importlib
import ast
import pandas as pd
import re

import shared.normalization as normalization
from shared.project_paths import get_project_paths

importlib.reload(normalization)
from shared.normalization import normalize_segment

paths = get_project_paths()
CODE_ROOT = paths.code_root
PROJECT_ROOT = paths.project_root
DATA_DIR = paths.data_dir
OUTPUT_DIR = paths.output_dir("003-extract-dm-candidates")

PAPERS_PATH = DATA_DIR / "papers.parquet"
PAPER_CLASSES_PATH = DATA_DIR / "paper_arxiv_classes.parquet"
ENRICHED_PARQUET = DATA_DIR / "papers_with_dm_models.parquet"
CANDIDATE_LONG_PARQUET = DATA_DIR / "dm_model_candidates_long.parquet"
MENTIONS_PARQUET = OUTPUT_DIR / "dm_model_mentions.parquet"
COUNTS_PARQUET = OUTPUT_DIR / "dm_model_counts_by_year.parquet"

EARLIEST_YEAR = 1950
END_YEAR = 2025
BAD_DOC = [
    'pressrelease', 'mastersthesis', 'misc', 'techreport', 'dataset', 'proposal', 'proceedings',
    'abstract', 'newsletter', 'obituary', 'talk', 'circular', 'bookreview', 'software',
    'editorial', 'erratum', 'phdthesis',
]
BIOLOGY_TERMS_STRICT = {
    'microbial', 'bacterial', 'genome', 'genomes', 'eukaryotic', 'genomic',
    'microbes', 'microorganisms', 'bacteria', 'metabolites', 'phyla', 'mags', 'metagenomics'
}
BIOLOGY_PATTERN = re.compile(r"\b(?:%s)\b" % '|'.join(map(re.escape, BIOLOGY_TERMS_STRICT)), flags=re.I)

if not PAPERS_PATH.exists():
    raise FileNotFoundError(f"Missing required input: {PAPERS_PATH}")
if not PAPER_CLASSES_PATH.exists():
    raise FileNotFoundError(f"Missing required input: {PAPER_CLASSES_PATH}")

papers = pd.read_parquet(PAPERS_PATH)
paper_classes = pd.read_parquet(PAPER_CLASSES_PATH)

class_lists = (
    paper_classes.sort_values(['bibcode', 'class_pos'], kind='mergesort')
    .groupby('bibcode', as_index=False)
    .agg(arxiv_class=('arxiv_class', list))
)

df = papers.merge(class_lists, on='bibcode', how='left')
df['arxiv_class'] = df['arxiv_class'].apply(lambda value: value if isinstance(value, list) else [])
df['year'] = pd.to_numeric(df['year'], errors='coerce').astype('Int64')
df['doctype'] = df['doctype'].astype('string')
df['abstract'] = df['abstract'].astype('string')

year_mask = df['year'].notna() & df['year'].between(EARLIEST_YEAR, END_YEAR, inclusive='both')
abstract_mask = df['abstract'].notna()
doc_mask = ~df['doctype'].isin(BAD_DOC)
biology_mask = ~df['abstract'].str.contains(BIOLOGY_PATTERN, na=False)

eligible_mask = year_mask & abstract_mask & doc_mask & biology_mask

df['eligible_for_model_extraction'] = eligible_mask
df['abstract_clean'] = pd.Series(pd.NA, index=df.index, dtype='string')
df.loc[eligible_mask, 'abstract_clean'] = (
    df.loc[eligible_mask, 'abstract']
      .astype('string')
      .str.lower()
      .map(lambda s: normalize_segment(s, mask_math=True))
      .astype('string')
)

print(f"Loaded papers: {len(df):,}")
print(f"Rows with abstracts: {abstract_mask.sum():,}")
print(f"Rows eligible for extraction: {eligible_mask.sum():,}")
print(f"Enriched parquet target: {ENRICHED_PARQUET}")
print(f"Candidate-long parquet target: {CANDIDATE_LONG_PARQUET}")


Loaded papers: 201,627
Rows with abstracts: 188,691
Rows eligible for extraction: 177,529
Enriched parquet target: /Users/sallz/damadi/Nature Astronomy/analysis-version/data/papers_with_dm_models.parquet
Candidate-long parquet target: /Users/sallz/damadi/Nature Astronomy/analysis-version/data/dm_model_candidates_long.parquet


## Stage 1. Candidate-pattern extraction rules


In [ ]:
import re
import unicodedata
import pandas as pd

# -------------------------
# Normalization
# -------------------------
def normalize_text(s: str) -> str:
    if not isinstance(s, str):
        return ""
    return normalize_segment(str(s).lower(), mask_math=True)

# -------------------------
# Canonical tag -> regex list
# (case-insensitive; include hyphen/space/plural variants)
# -------------------------
PATTERNS = {
    # --- WIMP family & SUSY ---
    "wimp": [r"\bwimp(?:s)?\b", r"\bweakly(?:[-\s])?interacting massive particle(?:s)?\b"],
    "neutralino": [r"\bneutralino(?:s)?\b"],
    "lsp": [r"\blightest supersymmetric particle(?:s)?\b", r"\bLSP(?:s)?\b"],
    "wino_dm": [r"\bwino(?:s)?(?:\s+dark\s+matter)?\b"],
    "higgsino_dm": [r"\bhiggsino(?:s)?(?:\s+dark\s+matter)?\b"],
    "bino_dm": [r"\bbino(?:s)?(?:\s+dark\s+matter)?\b"],
    "sneutrino_dm": [r"\bsneutrino(?:s)?(?:\s+dark\s+matter)?\b"],
    "axino_dm": [r"\baxino(?:s)?(?:\s+dark\s+matter)?\b"],
    "gravitino_dm": [r"\bgravitino(?:s)?(?:\s+dark\s+matter)?\b"],
    "superwimp": [r"\bsuper[-\s]?wimp(?:s)?\b", r"\bsuper\s*weakly\s*interacting massive particle(?:s)?\b", r"\bSWIMP(?:s)?\b"],

    # --- Axions / ALPs / ULAs / fuzzy / waveDM ---
    "axion": [r"\baxion(?:s)?\b", r"\bqcd\s+axion(?:s)?\b"],
    "alp": [r"\balp(?:s)?\b", r"\baxion(?:[-\s]?like)?\s+particle(?:s)?\b", r"\baxionlike\s+particle(?:s)?\b"],
    "ula": [r"\bula(?:s)?\b", r"\bultra[-\s]?light\s+axion(?:s)?\b"],
    "fuzzy_dm": [
        r"\bfuzzy\s+dark\s+matter\b", r"\bfdm\b",
        r"\bwave[-\s]?dm\b",
        r"\b(?:ψ|\u03c8)\s*dm\b",
        r"\bultra[-\s]?light\s+(?:scalar|boson)\s+dark\s+matter\b",
        r"\bbec\s+dark\s+matter\b"
    ],
    "scalar_field_dm": [r"\bscalar\s+field\s+dark\s+matter\b", r"\bSFDM\b"],
    "superfluid_dm": [r"\bsuperfluid\s+dark\s+matter\b"],

    # --- Sterile neutrinos ---
    "sterile_neutrino": [
        r"\bsterile[-\s]?neutrino(?:s)?\b",
        r"\bsterile\s+neutrino\s+dark\s+matter\b",
        r"\bDodelson[-\s]?Widrow\b",
        r"\bShi[-\s]?Fuller\b",
        r"\b\nu[_\s]?s\b", r"\bnu[_\s]?s\b"
    ],

    # --- PBHs / MACHOs / Macros ---
    "pbh": [r"\bpbh(?:s)?\b", r"\bprimordial\s+black\s+hole(?:s)?\b", r"\bPBH(?:s)?\b"],
    "macho": [r"\bmacho(?:s)?\b", r"\bmassive\s+compact\s+halo\s+object(?:s)?\b"],
    "macro_dm": [r"\bmacro(?:s)?\b", r"\bmacroscopic\s+dark\s+matter\b"],

    # --- Asymmetric / composite / atomic / mirror / twin ---
    "adm": [r"\basymmetric\s+dark\s+matter\b", r"\bADM\b"],
    "composite_dm": [r"\bcomposite\s+dark\s+matter\b", r"\bdark\s+(?:baryon|baryons|pion|pions|meson|mesons|hadron|hadrons)\b"],
    "atomic_dm": [r"\batomic\s+dark\s+matter\b", r"\bdark\s+atom(?:s)?\b", r"\bdark\s+hydrogen\b", r"\bO[-\s]?He(?:lium)?\b"],
    "mirror_dm": [r"\bmirror\s+(?:dark\s+)?matter\b", r"\bmirror\s+(?:baryon|baryons)\b"],
    "twin_higgs_dm": [r"\btwin\s+higgs\b", r"\btwin\s+baryon(?:s)?\s+dark\s+matter\b", r"\bmirror\s+twin\s+higgs\b"],

    # --- Interacting paradigms & production mechanisms (often used as candidate labels) ---
    "sidm": [r"\bself[-\s]?interacting\s+dark\s+matter\b", r"\bSIDM\b"],
    "wdm": [r"\bwarm\s+dark\s+matter\b", r"\bWDM\b"],
    "ldm": [r"\blight\s+dark\s+matter\b", r"\bLDM\b", r"\bsub[-\s]?GeV\s+dark\s+matter\b", r"\bMeV[-\s]?scale\s+dark\s+matter\b"],
    "simp": [r"\bstrongly\s+interacting\s+massive\s+particle(?:s)?\b", r"\bSIMP(?:s)?\b"],
    "elder": [r"\bELDER(?:\s*DM)?\b", r"\belastically\s+decoupling\s+relic\b"],
    "fimp": [r"\bfeebly\s+interacting\s+massive\s+particle(?:s)?\b", r"\bFIMP(?:s)?\b", r"\bfreeze[-\s]?in\b"],
    "forbidden_dm": [r"\bforbidden\s+dark\s+matter\b"],
    "secluded_dm": [r"\bsecluded\s+dark\s+matter\b"],
    "cannibal_dm": [r"\bcannibal\s+dark\s+matter\b"],
    "codecaying_dm": [r"\bco[-\s]?decay(?:ing)?\s+dark\s+matter\b"],
    "semiannihilating_dm": [r"\bsemi[-\s]?annihilat(?:ion|ing)\s+dark\s+matter\b"],
    "inelastic_dm": [r"\binelastic\s+dark\s+matter\b", r"\biDM\b", r"\bpseudo[-\s]?Dirac\b"],
    "excited_dm": [r"\bexcited\s+dark\s+matter\b", r"\bXDM\b"],
    "boosted_dm": [r"\bboosted\s+dark\s+matter\b", r"\bBDM\b"],
    "decaying_dm": [r"\bdecay(?:ing)?\s+dark\s+matter\b", r"\bdecay\s+of\s+dark\s+matter\b"],
    "self_heating_dm": [r"\bself[-\s]?heating\s+dark\s+matter\b"],
    "cointeracting_dm": [r"\bco[-\s]?interacting\s+dark\s+matter\b", r"\bCiDM\b"],

    # --- Portals/mediators when the mediator is the DM candidate ---
    "higgs_portal_dm": [r"\bhiggs\s+portal(?:\s+dark\s+matter)?\b", r"\bscalar\s+singlet(?:\s+dark\s+matter)?\b", r"\breal\s+singlet\s+scalar\b", r"\bcomplex\s+scalar\s+dark\s+matter\b", r"\bsinglet\s+scalar\s+dm\b"],
    "z_portal_dm": [r"\bz['’\-]?\s*portal\b", r"\bz[-\s]?portal\b"],
    "neutrino_portal_dm": [r"\bneutrino\s+portal(?:\s+dark\s+matter)?\b"],
    "vector_dm": [r"\bvector\s+dark\s+matter\b", r"\bVDM\b"],
    "dark_photon_dm": [
        r"\bdark\s+photon(?:s)?\b", r"\bhidden\s+photon(?:s)?\b", r"\bpara[-\s]?photon(?:s)?\b",
        r"\bA['′]?\s*prime\b", r"\bA['′]\b", r"\bZ[_-]?[dD]\b", r"\bdark\s+Z\b",
        r"\bU\(1\)['’]?\b", r"\bdark\s+U\(1\)\b", r"\bkinetic\s+mixing\b"
    ],

    # --- EM-coupled ---
    "millicharged_dm": [r"\bmilli[-\s]?charged\s+(?:particle|dark\s+matter)(?:s)?\b", r"\bmillicharged\b", r"\bmCP(?:s)?\b"],
    "anapole_dm": [r"\banapole\s+dark\s+matter\b"],
    "dipole_dm": [r"\b(?:magnetic|electric)\s+dipole\s+dark\s+matter\b", r"\brayleigh\s+dark\s+matter\b"],

    # --- Confined/hidden gauge sectors ---
    "glueball_dm": [r"\bglueball\s+dark\s+matter\b", r"\bdark\s+glueball(?:s)?\b", r"\bhidden[-\s]?glue\b"],
    "hidden_sector_dm": [r"\bhidden[-\s]?sector\s+(?:dark\s+)?matter\b", r"\bdark\s+sector\s+(?:dark\s+)?matter\b", r"\bsecluded\s+sector\b"],

    # --- Extra dimensions ---
    "kk_dm": [r"\bkaluza[-\s]?klein\s+(?:dark\s+)?matter\b", r"\bkk\s+dark\s+matter\b", r"\blightest\s+kaluza[-\s]?klein\s+particle\b", r"\bLKP\b"],
    "ued_dm": [r"\buniversal\s+extra\s+dimension(?:s)?\b", r"\bUED\b", r"\bB\(1\)\b"],

    # --- Superheavy / exotic composites ---
    "wimpzilla": [r"\bwimpzilla(?:s)?\b", r"\bsuperheavy\s+dark\s+matter\b"],
    "qball_dm": [r"\bq[-\s]?ball(?:s)?\b(?:.*?\bdark\s+matter\b)?"],
    "quark_nugget_dm": [r"\baxion\s+quark\s+nugget(?:s)?\b", r"\bAQN\b", r"\bquark\s+nugget(?:s)?\b", r"\bnuclearite(?:s)?\b", r"\bstrange\s+quark\s+matter\b", r"\bstrangelet(?:s)?\b"],

    # --- Model families commonly cited by name ---
    "minimal_dm": [r"\bminimal\s+dark\s+matter\b"],
    "inert_doublet_dm": [r"\binert\s+doublet\s+model\b", r"\bIDM\b"],
    "inert_triplet_dm": [r"\binert\s+triplet\s+model\b", r"\bITM\b"],
    "scotogenic_dm": [r"\bscotogenic\b"],
    "majoron_dm": [r"\bmajoron(?:s)?\b"],

    # --- Coupling-structure labels used as candidate types ---
    "leptophilic_dm": [r"\bleptophilic\s+dark\s+matter\b"],
    "leptophobic_dm": [r"\bleptophobic\s+dark\s+matter\b"],
    "hadrophilic_dm": [r"\bhadrophilic\s+dark\s+matter\b"],
    "isospin_violating_dm": [r"\bisospin[-\s]?violat(?:ing|ion)\s+dark\s+matter\b", r"\bIVDM\b"],
    "planckian_dm": [r"\bplanckian\s+interacting\s+(?:massive\s+)?particle(?:s)?\b", r"\bPIDM\b"],
    "partially_interacting_dm": [r"\bpartially\s+interacting\s+dark\s+matter\b"],
}

PATTERNS["alp"][0] = r"\bALP(?:s)?\b(?!\s*(?:II|III|experiment|collaboration))"
AMBIGUOUS_REQUIRES_DM_NEARBY = {"glueball_dm", "hidden_sector_dm", "ued_dm", "z_portal_dm", "dark_photon_dm", "kk_dm", "vector_dm", "millicharged_dm", "qball_dm", "majoron_dm", "scotogenic_dm", "inert_doublet_dm", "inert_triplet_dm", "minimal_dm", "higgs_portal_dm", "neutrino_portal_dm", "partially_interacting_dm"}
NEAR_DM = re.compile(r"\b(dark\s+matter|DM)\b", re.I)

def dm_nearby_in_window(text: str, span: tuple[int, int], window: int = 60) -> bool:
    a, b = max(0, span[0] - window), min(len(text), span[1] + window)
    return bool(NEAR_DM.search(text[a:b]))

COMPILED = {tag: [re.compile(p, re.I) for p in pats] for tag, pats in PATTERNS.items()}

def extract_dm_tags_with_spans(text: str, window: int = 60):
    """
    Returns canonical tags and spans in first-mention order using the cleaned abstract text.
    """
    t = text if isinstance(text, str) else ""
    if not t:
        return [], []

    hits = []
    for tag, regs in COMPILED.items():
        for rg in regs:
            m = rg.search(t)
            if m:
                if tag in AMBIGUOUS_REQUIRES_DM_NEARBY and not dm_nearby_in_window(t, m.span(), window):
                    break
                hits.append((m.start(), tag, m.span()))
                break

    seen = set()
    tags_in_order, spans = [], []
    for _, tag, span in sorted(hits, key=lambda x: x[0]):
        if tag not in seen:
            seen.add(tag)
            tags_in_order.append(tag)
            spans.append(span)
    return tags_in_order, spans

if "df" in globals() and isinstance(df, pd.DataFrame) and "abstract_clean" in df.columns:
    df[["dm_models", "dm_spans"]] = df["abstract_clean"].apply(
        lambda s: pd.Series(extract_dm_tags_with_spans(s))
    )
    print(df[["abstract_clean", "dm_models", "dm_spans"]].head(20))


## Stage 2. Candidate labels and paper-level fields


In [ ]:
# Normalization + candidate-only columns

import pandas as pd

DM_TAGS_INFO = {
    "wimp": {"full": "Weakly Interacting Massive Particle (WIMP)", "category": "candidate"},
    "neutralino": {"full": "Neutralino (SUSY LSP)", "category": "candidate"},
    "lsp": {"full": "Lightest Supersymmetric Particle (LSP)", "category": "candidate"},
    "wino_dm": {"full": "Wino", "category": "candidate"},
    "higgsino_dm": {"full": "Higgsino", "category": "candidate"},
    "bino_dm": {"full": "Bino", "category": "candidate"},
    "sneutrino_dm": {"full": "Sneutrino", "category": "candidate"},
    "axino_dm": {"full": "Axino", "category": "candidate"},
    "gravitino_dm": {"full": "Gravitino", "category": "candidate"},
    "superwimp": {"full": "SuperWIMP", "category": "candidate"},
    "axion": {"full": "QCD Axion", "category": "candidate"},
    "alp": {"full": "Axion-Like Particle (ALP)", "category": "candidate"},
    "ula": {"full": "Ultralight Axion (ULA)", "category": "candidate"},
    "fuzzy_dm": {"full": "Fuzzy / waveDM", "category": "candidate"},
    "scalar_field_dm": {"full": "Scalar Field (SFDM)", "category": "candidate"},
    "superfluid_dm": {"full": "Superfluid", "category": "candidate"},
    "sterile_neutrino": {"full": "Sterile Neutrino", "category": "candidate"},
    "pbh": {"full": "Primordial Black Hole (PBH)", "category": "candidate"},
    "macho": {"full": "Massive Compact Halo Object (MACHO)", "category": "candidate"},
    "macro_dm": {"full": "Macroscopic (Macros)", "category": "candidate"},
    "quark_nugget_dm": {"full": "Quark Nugget / Nuclearite / Strangelet (AQN)", "category": "candidate"},
    "adm": {"full": "Asymmetric (ADM)", "category": "candidate"},
    "composite_dm": {"full": "Composite", "category": "candidate"},
    "atomic_dm": {"full": "Atomic", "category": "candidate"},
    "mirror_dm": {"full": "Mirror Sector", "category": "candidate"},
    "twin_higgs_dm": {"full": "Twin Higgs", "category": "candidate"},
    "glueball_dm": {"full": "Glueball", "category": "candidate"},
    "dark_photon_dm": {"full": "Dark Photon / Hidden Photon", "category": "candidate"},
    "vector_dm": {"full": "Vector", "category": "candidate"},
    "higgs_portal_dm": {"full": "Higgs Portal", "category": "portal"},
    "neutrino_portal_dm": {"full": "Neutrino Portal", "category": "portal"},
    "z_portal_dm": {"full": "Z/Z′ Portal", "category": "portal"},
    "hidden_sector_dm": {"full": "Hidden Sector / Dark Sector", "category": "umbrella"},
    "secluded_dm": {"full": "Secluded", "category": "portal"},
    "kk_dm": {"full": "Kaluza–Klein", "category": "candidate"},
    "ued_dm": {"full": "Universal Extra Dimensions (LKP)", "category": "candidate"},
    "wimpzilla": {"full": "WIMPzilla", "category": "candidate"},
    "planckian_dm": {"full": "Planckian Interacting (PIDM)", "category": "candidate"},
    "partially_interacting_dm": {"full": "Partially Interacting", "category": "interaction"},
    "qball_dm": {"full": "Q-ball", "category": "candidate"},
    "majoron_dm": {"full": "Majoron", "category": "candidate"},
    "minimal_dm": {"full": "Minimal SU(2)_L multiplet", "category": "candidate"},
    "inert_doublet_dm": {"full": "Inert Doublet Model (IDM)", "category": "candidate"},
    "inert_triplet_dm": {"full": "Inert Triplet Model (ITM)", "category": "candidate"},
    "scotogenic_dm": {"full": "Scotogenic Model", "category": "candidate"},
    "wdm": {"full": "Warm (WDM)", "category": "umbrella"},
    "ldm": {"full": "Light (sub-GeV/MeV-scale)", "category": "umbrella"},
    "sidm": {"full": "Self-Interacting (SIDM)", "category": "interaction"},
    "inelastic_dm": {"full": "Inelastic (iDM)", "category": "interaction"},
    "excited_dm": {"full": "Excited (XDM)", "category": "interaction"},
    "boosted_dm": {"full": "Boosted (BDM)", "category": "interaction"},
    "simp": {"full": "Strongly Interacting Massive Particle (SIMP)", "category": "interaction"},
    "fimp": {"full": "Feebly Interacting Massive Particle (FIMP)", "category": "mechanism"},
    "elder": {"full": "ELDER", "category": "mechanism"},
    "forbidden_dm": {"full": "Forbidden-Channel", "category": "mechanism"},
    "codecaying_dm": {"full": "Co-decaying", "category": "mechanism"},
    "cointeracting_dm": {"full": "Co-interacting", "category": "interaction"},
    "semiannihilating_dm": {"full": "Semi-annihilating", "category": "interaction"},
    "cannibal_dm": {"full": "Cannibal", "category": "mechanism"},
    "self_heating_dm": {"full": "Self-heating", "category": "mechanism"},
    "decaying_dm": {"full": "Decaying", "category": "mechanism"},
    "isospin_violating_dm": {"full": "Isospin-Violating (IVDM)", "category": "interaction"},
    "leptophilic_dm": {"full": "Leptophilic", "category": "interaction"},
    "leptophobic_dm": {"full": "Leptophobic", "category": "interaction"},
    "hadrophilic_dm": {"full": "Hadrophilic", "category": "interaction"},
    "millicharged_dm": {"full": "Millicharged", "category": "interaction"},
    "dipole_dm": {"full": "Dipole-Interacting", "category": "interaction"},
    "anapole_dm": {"full": "Anapole", "category": "interaction"},
}

CANDIDATE_TAGS = {t for t, info in DM_TAGS_INFO.items() if info.get("category") == "candidate"}
SPECIES_LABEL_MAP = {
    "wimp": "Generic WIMP",
    "neutralino": "Neutralino",
    "lsp": "LSP",
    "wino_dm": "Wino",
    "higgsino_dm": "Higgsino",
    "bino_dm": "Bino",
    "sneutrino_dm": "Sneutrino",
    "axino_dm": "Axino",
    "gravitino_dm": "Gravitino",
    "superwimp": "SuperWIMP",
    "axion": "Axion + ALP",
    "alp": "Axion + ALP",
    "ula": "ULA / Fuzzy",
    "fuzzy_dm": "ULA / Fuzzy",
    "scalar_field_dm": "SFDM",
    "superfluid_dm": "Superfluid DM",
    "sterile_neutrino": "Sterile ν",
    "pbh": "PBH",
    "macho": "MACHO",
    "macro_dm": "Macroscopic DM",
    "quark_nugget_dm": "Quark Nugget / AQN",
    "adm": "Asymmetric DM",
    "composite_dm": "Generic Composite",
    "atomic_dm": "Atomic DM",
    "mirror_dm": "Mirror DM",
    "twin_higgs_dm": "Twin Higgs",
    "glueball_dm": "Glueball",
    "dark_photon_dm": "Dark Photon",
    "vector_dm": "Vector DM",
    "kk_dm": "LKP Kaluza-Klein",
    "ued_dm": "LKP Kaluza-Klein",
    "wimpzilla": "WIMPzilla",
    "planckian_dm": "Planckian DM",
    "qball_dm": "Q-ball",
    "majoron_dm": "Majoron",
    "minimal_dm": "Minimal SU(2)L",
    "inert_doublet_dm": "Inert Doublet Model",
    "inert_triplet_dm": "Inert Triplet Model",
    "scotogenic_dm": "Scotogenic Model",
}

def expand_dm_tags(tags):
    if not isinstance(tags, (list, tuple)):
        return []
    return [DM_TAGS_INFO.get(t, {}).get("full", t) for t in tags]

def filter_candidate_tags(tags):
    if not isinstance(tags, (list, tuple)):
        return []
    return [t for t in tags if t in CANDIDATE_TAGS]

def candidate_species_labels(tags):
    if not isinstance(tags, (list, tuple)):
        return []
    labels = []
    seen = set()
    for tag in tags:
        label = SPECIES_LABEL_MAP.get(tag, DM_TAGS_INFO.get(tag, {}).get("full", tag))
        if label not in seen:
            seen.add(label)
            labels.append(label)
    return labels

def primary_arxiv_class(classes):
    if isinstance(classes, (list, tuple)):
        for value in classes:
            if isinstance(value, str) and value.strip():
                return value.strip()
    return None

def categorize_arxiv(arxiv_class):
    if not isinstance(arxiv_class, str):
        return "Other"
    if re.match(r"^astro", arxiv_class):
        return "astrophysics"
    if re.match(r"^hep-", arxiv_class):
        return "high-energy physics"
    if re.match(r"^cond-mat", arxiv_class):
        return "condensed matter"
    if re.match(r"^math", arxiv_class):
        return "mathematics"
    if re.match(r"^cs", arxiv_class):
        return "computer science"
    if re.match(r"^quant-ph", arxiv_class):
        return "quantum physics"
    if re.match(r"^gr-qc", arxiv_class):
        return "general relativity and quantum cosmology"
    if re.match(r"^nucl-", arxiv_class):
        return "nuclear physics"
    if re.match(r"^physics", arxiv_class):
        return "physics"
    if re.match(r"^q-bio", arxiv_class):
        return "quantitative biology"
    if re.match(r"^q-fin", arxiv_class):
        return "quantitative finance"
    if re.match(r"^stat", arxiv_class):
        return "statistics"
    if re.match(r"^econ", arxiv_class):
        return "economics"
    if re.match(r"^eess", arxiv_class):
        return "electrical engineering and systems science"
    if re.match(r"^nlin", arxiv_class):
        return "nonlinear sciences"
    return "Other"

if "df" in globals() and isinstance(df, pd.DataFrame) and "dm_models" in df.columns:
    df["primary_arxiv_class"] = df["arxiv_class"].apply(primary_arxiv_class)
    df["arxiv_category"] = df["primary_arxiv_class"].apply(categorize_arxiv)
    df["dm_models_full"] = df["dm_models"].apply(expand_dm_tags)
    df["dm_models_candidates"] = df["dm_models"].apply(filter_candidate_tags)
    df["dm_models_candidates_full"] = df["dm_models_candidates"].apply(expand_dm_tags)
    df["dm_candidate_species"] = df["dm_models_candidates"].apply(candidate_species_labels)
    df["dm_models_count"] = df["dm_models"].apply(lambda L: len(L) if isinstance(L, list) else 0)
    df["dm_models_candidates_count"] = df["dm_models_candidates"].apply(len)


## Stage 3. Mention-level audit table


In [ ]:
import ast
import pandas as pd

def ensure_list(obj):
    if isinstance(obj, list):
        return obj
    if pd.isna(obj):
        return []
    if isinstance(obj, str):
        try:
            v = ast.literal_eval(obj)
            return v if isinstance(v, list) else [obj]
        except Exception:
            return [obj]
    if isinstance(obj, tuple):
        return list(obj)
    return [obj]

df["dm_models_candidates_full"] = df["dm_models_candidates_full"].apply(ensure_list)
df["dm_spans"] = df["dm_spans"].apply(ensure_list)

df["dm_spans"] = df["dm_spans"].apply(lambda L: [tuple(x) for x in L])

def pair_tags_spans(row):
    tags, spans = row["dm_models_candidates_full"], row["dm_spans"]
    n = min(len(tags), len(spans))
    return list(zip(tags[:n], spans[:n]))

df["dm_pairs"] = df.apply(pair_tags_spans, axis=1)

tidy = df.explode("dm_pairs", ignore_index=True)
tidy = tidy.dropna(subset=["dm_pairs"])
tidy[["tag", "span"]] = tidy["dm_pairs"].apply(pd.Series)
tidy = tidy.drop(columns=["dm_pairs"])

def make_snippet(text, span, radius=30):
    if not isinstance(text, str) or not isinstance(span, (list, tuple)):
        return ""
    a, b = span
    a, b = max(0, a - radius), min(len(text), b + radius)
    return text[a:b]

tidy["snippet"] = tidy.apply(
    lambda r: make_snippet(r["abstract_clean"] if isinstance(r.get("abstract_clean"), str) else "", r["span"]),
    axis=1,
)

print(tidy[["tag", "span", "snippet"]].head(20))


## Stage 4. Write the canonical and audit outputs


In [ ]:
df_unique = tidy.drop_duplicates(subset=['bibcode', 'tag', 'year'])
df_unique = df_unique[df_unique['tag'] != 'dark matter']

paper_level_cols = [
    'bibcode', 'year', 'arxiv_class', 'primary_arxiv_class', 'arxiv_category',
    'eligible_for_model_extraction', 'abstract_clean',
    'dm_models', 'dm_models_candidates', 'dm_candidate_species',
    'dm_models_count', 'dm_models_candidates_count'
]
papers_with_models = df[paper_level_cols].copy()
papers_with_models.to_parquet(ENRICHED_PARQUET, index=False)

mention_cols = ['bibcode', 'year', 'arxiv_category', 'tag', 'span', 'snippet']
dm_mentions = df_unique[mention_cols].copy()
dm_mentions.to_parquet(MENTIONS_PARQUET, index=False)

candidate_long = (
    papers_with_models[['bibcode', 'year', 'arxiv_category', 'dm_models_candidates', 'dm_candidate_species']]
    .assign(candidate_pairs=lambda d: d.apply(
        lambda row: list(zip(row['dm_models_candidates'], row['dm_candidate_species']))
        if isinstance(row['dm_models_candidates'], list) and isinstance(row['dm_candidate_species'], list)
        else [],
        axis=1
    ))
    .drop(columns=['dm_models_candidates', 'dm_candidate_species'])
    .explode('candidate_pairs', ignore_index=True)
)
candidate_long = candidate_long.dropna(subset=['candidate_pairs']).copy()
candidate_long[['tag', 'SpeciesLabel']] = candidate_long['candidate_pairs'].apply(pd.Series)
candidate_long = candidate_long.drop(columns=['candidate_pairs']).drop_duplicates(
    subset=['bibcode', 'year', 'arxiv_category', 'tag', 'SpeciesLabel']
).reset_index(drop=True)
candidate_long.to_parquet(CANDIDATE_LONG_PARQUET, index=False)

paper_count_per_year = (
    df_unique.groupby(['tag', 'year'])
    .size()
    .reset_index(name='paper_count')
)
paper_count_per_year.to_parquet(COUNTS_PARQUET, index=False)

print(f"Saved lean paper-level parquet -> {ENRICHED_PARQUET}")
print(f"Saved mention-level parquet -> {MENTIONS_PARQUET}")
print(f"Saved candidate-long parquet -> {CANDIDATE_LONG_PARQUET}")
print(f"Saved yearly tag counts -> {COUNTS_PARQUET}")
print(f"Candidate rows: {len(candidate_long):,}")
print(f"Mention rows: {len(dm_mentions):,}")
print(f"Unique candidate labels: {candidate_long['SpeciesLabel'].nunique():,}")
display(candidate_long.head(10))
